<a href="https://colab.research.google.com/github/reelelsadig/beginning-bioinformatics2/blob/main/GWAS_Pop_Struct.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Code to assess the effect of population structure on GWAS

In [ ]:
# Population Structure-adjusted logistic GWAS: Trait ~ Genotype + PC1 + PC2
# Saves association results and draws QQ and Manhattan plots.

import warnings
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
from scipy.stats import chi2

NUM_PCS = 2
MIN_SAMPLES = 10
MAX_ITER = 200

# ---- 0) Align samples and phenotypes ----
if "vcf" in globals() and hasattr(vcf, "samples"):
    samples = list(vcf.samples)
else:
    fixed_cols = {"CHROM", "POS", "ID", "REF", "ALT"}
    samples = [c for c in geno.columns if c not in fixed_cols]

if "IID" in phenos.columns:
    ph_series = phenos.set_index("IID")["Trait"].reindex(samples)
else:
    ph_series = phenos["Trait"].reindex(samples)

ph_series = pd.to_numeric(ph_series, errors="raise")

if not ph_series.dropna().isin([0, 1]).all():
    raise ValueError("Trait must be coded 0/1, with missing values allowed.")

if ph_series.dropna().nunique() < 2:
    raise ValueError("Trait must contain both cases (1) and controls (0).")

# ---- 1) Compute PCs: individuals x SNPs ----
G = geno[samples].T.to_numpy(dtype=float)

if np.isinf(G).any():
    raise ValueError("Genotypes contain infinite values.")

# Remove entirely missing SNPs before mean imputation
G = G[:, ~np.isnan(G).all(axis=0)]

if G.shape[1] == 0:
    raise ValueError("No observed genotypes are available for PCA.")

# Mean-impute missing genotypes for PCA only
col_mean = np.nanmean(G, axis=0)
inds = np.where(np.isnan(G))
G[inds] = col_mean[inds[1]]

# Remove monomorphic SNPs and standardize the remaining SNPs
std = G.std(axis=0, ddof=0)
variable = std > 0
G = G[:, variable]
std = std[variable]

if min(G.shape) < NUM_PCS:
    raise ValueError("Too few samples or variable SNPs to compute the PCs.")

X = (G - G.mean(axis=0)) / std
U, S, Vt = np.linalg.svd(X, full_matrices=False)

tol = S[0] * max(X.shape) * np.finfo(float).eps
if np.sum(S > tol) < NUM_PCS:
    raise ValueError("Genotype matrix has insufficient rank for the PCs.")

# Scale PC columns to unit standard deviation for numerical stability.
# This changes PC coefficient units, not the fitted genotype effect.
PCs = U[:, :NUM_PCS]
PCs = PCs / PCs.std(axis=0, ddof=0)

covar = pd.DataFrame(
    PCs,
    index=samples,
    columns=[f"PC{i+1}" for i in range(NUM_PCS)]
)

print(
    f"Computed PCs: shape {PCs.shape}. "
    f"Using PC1–PC{NUM_PCS} as covariates."
)

# ---- 2) Logistic GWAS with PCs ----
def safe_exp(value):
    """Return infinity when exp(value) exceeds floating-point range."""
    if value > np.log(np.finfo(float).max):
        return np.inf
    with np.errstate(under="ignore"):
        return float(np.exp(value))

assoc_rows = []
excluded_rows = []
n_tests = 0

for _, row in geno.iterrows():
    snp = row.get("ID")
    if pd.isna(snp) or str(snp) in {"", "."}:
        snp = f"{row['CHROM']}_{row['POS']}"

    meta = {
        "SNP": str(snp),
        "CHR": str(row["CHROM"]),
        "BP": int(row["POS"])
    }

    g = row[samples].astype(float)
    m = g.notna() & ph_series.notna()
    g_eff = g[m]
    y = ph_series[m]

    if m.sum() < MIN_SAMPLES:
        excluded_rows.append({
            **meta, "REASON": "Too few usable samples"
        })
        continue

    if g_eff.nunique() < 2:
        excluded_rows.append({
            **meta, "REASON": "Monomorphic after masking"
        })
        continue

    if y.nunique() < 2:
        excluded_rows.append({
            **meta, "REASON": "Only one phenotype class after masking"
        })
        continue

    # Keep the number of attempted tests for the Bonferroni threshold,
    # including models that subsequently fail.
    n_tests += 1

    C = covar.loc[g_eff.index].to_numpy()
    Xdesign = np.column_stack([
        np.ones(len(g_eff)), g_eff.to_numpy(), C
    ])

    try:
        if np.linalg.matrix_rank(Xdesign) < Xdesign.shape[1]:
            raise ValueError("Design matrix is rank deficient")

        # Capture fit warnings and inspect them explicitly.
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            fit = sm.Logit(
                y.to_numpy(dtype=int), Xdesign
            ).fit(disp=False, maxiter=MAX_ITER)

        if not fit.mle_retvals.get("converged", False):
            raise ValueError("Model did not converge")

        warning_names = {w.category.__name__ for w in caught}
        if warning_names & {
            "PerfectSeparationWarning",
            "HessianInversionWarning"
        }:
            raise ValueError(
                "Separation or unavailable coefficient uncertainty"
            )

        beta = float(fit.params[1])
        se = float(fit.bse[1])
        pval = float(fit.pvalues[1])

        if not np.all(np.isfinite([beta, se, pval])):
            raise ValueError("Nonfinite coefficient, SE, or P value")

        if se <= 0 or not 0 <= pval <= 1:
            raise ValueError("Invalid SE or P value")

        assoc_rows.append({
            **meta,
            "N": int(m.sum()),
            "P": pval,
            "BETA": beta,
            "SE": se,
            "OR": safe_exp(beta),
            "OR_L95": safe_exp(beta - 1.96 * se),
            "OR_U95": safe_exp(beta + 1.96 * se),
            "FIT_WARNINGS": "; ".join(
                dict.fromkeys(str(w.message) for w in caught)
            )
        })

    except Exception as err:
        excluded_rows.append({**meta, "REASON": str(err)})

result_cols = [
    "SNP", "CHR", "BP", "N", "P", "BETA", "SE",
    "OR", "OR_L95", "OR_U95", "FIT_WARNINGS"
]
assoc_withPC = pd.DataFrame(assoc_rows, columns=result_cols)
excluded_withPC = pd.DataFrame(
    excluded_rows, columns=["SNP", "CHR", "BP", "REASON"]
)

assoc_withPC.to_csv("assoc_withPC.tsv", sep="\t", index=False)
excluded_withPC.to_csv(
    "assoc_withPC_excluded.tsv", sep="\t", index=False
)

print(f"[*] Wrote assoc_withPC.tsv with {len(assoc_withPC)} variants")
print(f"[*] Attempted {n_tests} SNP tests")
print(f"[*] Excluded/skipped {len(excluded_withPC)} variants")

if not excluded_withPC.empty:
    print("\nExclusion summary:")
    print(excluded_withPC["REASON"].value_counts().to_string())
    print("\nDetails saved to assoc_withPC_excluded.tsv")

if assoc_withPC.empty:
    raise ValueError("No valid PC-adjusted association results to plot.")

n_warned = assoc_withPC["FIT_WARNINGS"].ne("").sum()
if n_warned:
    print(
        f"[*] {n_warned} retained fits have additional warnings; "
        "review FIT_WARNINGS in the results."
    )

n_unbounded = np.isinf(assoc_withPC["OR_U95"]).sum()
if n_unbounded:
    print(
        f"[*] {n_unbounded} upper OR confidence limits exceed numerical "
        "range and are reported as infinity; inspect these estimates."
    )

# ---- 3) QQ plot with genomic inflation lambda ----
def lambda_gc(p):
    p = np.asarray(p, dtype=float)
    p = p[np.isfinite(p)]
    if p.size == 0:
        return np.nan
    chisq = chi2.isf(np.clip(p, 1e-300, 1), df=1)
    return np.median(chisq) / chi2.ppf(0.5, df=1)

lam = lambda_gc(assoc_withPC["P"].to_numpy())

p = np.sort(np.clip(assoc_withPC["P"].to_numpy(), 1e-300, 1))
expected_p = (np.arange(1, len(p) + 1) - 0.5) / len(p)
exp = -np.log10(expected_p)
obs = -np.log10(p)

plt.figure(figsize=(5, 5))
plt.scatter(exp, obs, s=10)
mx = max(exp.max(), obs.max())
plt.plot([0, mx], [0, mx], color="gray", lw=1)
plt.xlabel("Expected -log10(P)")
plt.ylabel("Observed -log10(P)")
plt.title(f"QQ plot with PCs (λ={lam:.2f})")
plt.tight_layout()
plt.show()

# ---- 4) Manhattan plot ----
plot_df = assoc_withPC.copy()
plot_df["neglog10P"] = -np.log10(
    plot_df["P"].clip(lower=1e-300)
)

bonf_p = 0.05 / n_tests
bonf_y = -np.log10(bonf_p)

def chr_key(c):
    label = str(c).lower().removeprefix("chr")
    return (0, int(label)) if label.isdigit() else (1, label)

chrom_order = sorted(plot_df["CHR"].unique(), key=chr_key)
fig, ax = plt.subplots(
    figsize=(9, 3.5) if len(chrom_order) == 1 else (10, 4)
)

if len(chrom_order) == 1:
    ax.scatter(plot_df["BP"], plot_df["neglog10P"], s=14)
    ax.set_xlabel(f"Position on chromosome {chrom_order[0]} (bp)")

else:
    # Offset each chromosome by the preceding chromosomes' maximum
    # observed positions, avoiding overlapping chromosome coordinates.
    offset = 0
    tick_positions = []
    colors = ["#2878B5", "#78B7C5"]

    for i, chrom in enumerate(chrom_order):
        chr_df = plot_df.loc[
            plot_df["CHR"] == chrom
        ].sort_values("BP")

        positions = chr_df["BP"] + offset
        ax.scatter(
            positions,
            chr_df["neglog10P"],
            s=10,
            color=colors[i % len(colors)]
        )
        tick_positions.append(
            (positions.min() + positions.max()) / 2
        )
        offset += chr_df["BP"].max() + 1

    ax.set_xticks(tick_positions)
    ax.set_xticklabels(chrom_order)
    ax.set_xlabel("Chromosome")

ax.axhline(
    bonf_y,
    ls="--",
    lw=1,
    color="gray",
    label=f"Bonferroni: P < {bonf_p:.2g}"
)
ax.set_ylabel("-log10(P)")
ax.set_title("Manhattan plot (PC-adjusted)")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()